# Gaussian Process Forecasting & Anomaly Detection
### Applied to Manufacturing Cycle Times

---

## 1. What is a Gaussian Process?

A **Gaussian Process (GP)** is a probability distribution over *functions*. Instead of fitting a single curve, a GP fits a whole *family* of plausible curves and tells you how confident it is everywhere.

### Core Analogy
> "A GP is to functions what a Gaussian distribution is to numbers."

A normal Gaussian describes a single number with mean mu and variance sigma^2.  
A GP describes a distribution over an entire **function** with a **mean function** m(x) and a **kernel (covariance) function** k(x, x').

Any finite set of function values is jointly Gaussian:  
**f ~ N(m, K)** where K_ij = k(x_i, x_j) — the kernel matrix.

### The Rubber Band Analogy
- Points **close together** → band is taut → predictions tightly correlated  
- Points **far apart** → band is slack → predictions grow uncertain  
The **kernel** controls how this tension decays with distance.

### GP Regression

Given training data (X, y) with noise eps ~ N(0, sigma_n^2), the posterior at test points X_* is:

**Posterior mean** (prediction):  
f_bar = K_*X (K_XX + sigma_n^2 I)^{-1} y

**Posterior variance** (uncertainty):  
cov(f_*) = K_** - K_*X (K_XX + sigma_n^2 I)^{-1} K_X*

**Key insight**: Variance shrinks near data points and grows in gaps — automatically, for free.

### Kernels

| Kernel | Behaviour | Use when... |
|---|---|---|
| **RBF** | Very smooth | Stable, smooth processes |
| **Matern 3/2** | Rough, 1x differentiable | Real-world data with jumps |
| **Matern 5/2** | Between RBF and Matern 3/2 | Good default |
| **Periodic + RBF** | Repeating + smooth trend | Weekly seasonality |
| **RationalQuadratic** | Multi-scale smoothness | Mix of length scales |

Kernels can be **combined**: k1 + k2 (additive) or k1 * k2 (multiplicative).

### GP Anomaly Detection

Once fit, every point is scored: z = (y - f_bar) / sqrt(var(f))  
Points with |z| > threshold are anomalies — outside what the model expected. Unlike a simple threshold, GP detection *adapts* to local trends and uncertainty.

---
## 2. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, WhiteKernel, ConstantKernel as C,
    ExpSineSquared, RationalQuadratic
)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 11
print('All imports OK')

---
## 3. Load & Explore Data

In [ ]:
df_raw = pd.read_csv('cycle_times.csv')
df_raw['date'] = pd.to_datetime(df_raw['date'])
print(f"Records: {len(df_raw):,}")
print(f"Date range: {df_raw['date'].min().date()} to {df_raw['date'].max().date()}")
print(f"Unique IDs: {df_raw['id'].nunique():,}")
print("\ncycle_time_s summary:")
print(df_raw['cycle_time_s'].describe().round(3))

In [ ]:
daily = df_raw.groupby('date')['cycle_time_s'].agg(
    count='count', mean='mean', median='median', std='std',
    p5=lambda x: x.quantile(0.05),
    p95=lambda x: x.quantile(0.95),
    p99=lambda x: x.quantile(0.99)
).reset_index()
daily['dow'] = daily['date'].dt.day_name()
daily['is_weekend'] = daily['date'].dt.dayofweek >= 5

fig, axes = plt.subplots(3, 1, figsize=(14, 11))

ax = axes[0]
ax.fill_between(daily['date'], daily['p5'], daily['p95'], alpha=0.2, label='5th-95th pct')
ax.plot(daily['date'], daily['mean'], 'o-', lw=2, label='Daily mean', ms=5)
ax.plot(daily['date'], daily['median'], 's--', lw=1.5, label='Daily median', ms=4, alpha=0.7)
for _, row in daily[daily['is_weekend']].iterrows():
    ax.axvspan(row['date'] - pd.Timedelta('0.4d'), row['date'] + pd.Timedelta('0.4d'), alpha=0.08, color='orange')
ax.set_title('Daily Cycle Time: Mean, Median & 5th-95th Percentile Band')
ax.set_ylabel('Cycle Time (s)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

ax = axes[1]
ax.bar(daily['date'], daily['count'],
       color=['orange' if w else 'steelblue' for w in daily['is_weekend']],
       alpha=0.8, width=0.8)
ax.set_title('Daily Record Count  (orange = weekend)')
ax.set_ylabel('Count')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

ax = axes[2]
ax.plot(daily['date'], daily['p99'], 'r-o', ms=4, label='p99')
ax.plot(daily['date'], daily['p95'], 'darkorange', ls='--', lw=1.5, label='p95')
ax.set_title('Daily Tail Latency (p95 & p99)')
ax.set_ylabel('Cycle Time (s)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

for ax in axes:
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 4. Configuration — Edit Here

All tunable parameters live in this one cell. Re-run cells below after making changes.

In [ ]:
# ========================================================
#          MAIN CONFIGURATION — EDIT HERE
# ========================================================

# 1. DATE RANGE (None = use all data)
DATE_START = '2026-02-02'   # 'YYYY-MM-DD' or None
DATE_END   = '2026-03-09'   # 'YYYY-MM-DD' or None

# 2. AGGREGATION:  'mean' | 'median' | 'p95' | 'p99'
AGGREGATE = 'mean'

# 3. OUTLIER REMOVAL: drop raw rows above this value before aggregating (None = keep all)
# OUTLIER_CUTOFF_S = 60.0
OUTLIER_CUTOFF_S = None

# 4. KERNEL:  'rbf' | 'matern32' | 'matern52' | 'periodic_rbf' | 'rq'
#   rbf          -> very smooth curves
#   matern32     -> allows roughness (good for real-world processes)
#   matern52     -> good default, between rbf and matern32
#   periodic_rbf -> weekly seasonality + smooth trend
#   rq           -> multi-scale (rational quadratic)
KERNEL_CHOICE = 'matern52'

# 5. FORECAST: how many days ahead to predict
FORECAST_DAYS = 7

# 6. CONFIDENCE INTERVAL coverage (0.95 = 95%)
CI_COVERAGE = 0.95

# 7. ANOMALY threshold: flag if |z-score| > this
ANOMALY_Z_THRESHOLD = 2.0

# 8. NORMALISE y before fitting? (usually helps optimisation)
NORMALISE_Y = True

# 9. Optimiser restarts (more = better but slower)
N_RESTARTS = 10

# ── Derived ──────────────────────────────────────────
CI_Z = stats.norm.ppf(0.5 + CI_COVERAGE / 2)

print(f"Config loaded:")
print(f"  Date range : {DATE_START} to {DATE_END}")
print(f"  Aggregate  : {AGGREGATE}")
print(f"  Kernel     : {KERNEL_CHOICE}")
print(f"  Forecast   : {FORECAST_DAYS} days")
print(f"  CI         : {CI_COVERAGE*100:.0f}% ({CI_Z:.2f} sigma)")
print(f"  Anomaly z  : |z| > {ANOMALY_Z_THRESHOLD}")

---
## 5. Prepare Data

In [ ]:
# Filter dates
df = df_raw.copy()
if DATE_START:
    df = df[df['date'] >= DATE_START]
if DATE_END:
    df = df[df['date'] <= DATE_END]

# Remove outliers
if OUTLIER_CUTOFF_S:
    n_before = len(df)
    df = df[df['cycle_time_s'] <= OUTLIER_CUTOFF_S]
    print(f"Outlier removal: dropped {n_before - len(df):,} rows > {OUTLIER_CUTOFF_S}s")

# Aggregate per day
agg_funcs = {
    'mean':   lambda x: x.mean(),
    'median': lambda x: x.median(),
    'p95':    lambda x: x.quantile(0.95),
    'p99':    lambda x: x.quantile(0.99),
}
series = df.groupby('date')['cycle_time_s'].apply(agg_funcs[AGGREGATE]).reset_index()
series.columns = ['date', 'y']
series = series.sort_values('date').reset_index(drop=True)

# Numeric X = days since first date
t0 = series['date'].min()
series['x'] = (series['date'] - t0).dt.days.astype(float)

X_train = series['x'].values.reshape(-1, 1)
y_train = series['y'].values

# Normalise y
if NORMALISE_Y:
    y_mean = y_train.mean()
    y_std  = y_train.std()
    y_scaled = (y_train - y_mean) / y_std
else:
    y_mean, y_std = 0.0, 1.0
    y_scaled = y_train

print(f"Training points : {len(X_train)}")
print(f"x range         : {X_train.min():.0f} to {X_train.max():.0f} days")
print(f"y ({AGGREGATE}) : {y_train.min():.3f} to {y_train.max():.3f}")

---
## 6. Define & Fit the Gaussian Process

In [ ]:
def build_kernel(choice):
    WK = WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-3, 1.0))
    if choice == 'rbf':
        k = C(1.0, (0.1, 10)) * RBF(length_scale=5, length_scale_bounds=(1, 50))
    elif choice == 'matern32':
        k = C(1.0, (0.1, 10)) * Matern(length_scale=5, length_scale_bounds=(1, 50), nu=1.5)
    elif choice == 'matern52':
        k = C(1.0, (0.1, 10)) * Matern(length_scale=5, length_scale_bounds=(1, 50), nu=2.5)
    elif choice == 'periodic_rbf':
        seasonal = C(0.5, (0.01, 5)) * ExpSineSquared(
            length_scale=2.0, periodicity=7.0,
            length_scale_bounds=(0.5, 10),
            periodicity_bounds=(6.5, 7.5))
        trend = C(1.0, (0.1, 10)) * RBF(length_scale=14, length_scale_bounds=(5, 60))
        k = seasonal + trend
    elif choice == 'rq':
        k = C(1.0, (0.1, 10)) * RationalQuadratic(length_scale=5, alpha=0.5,
                                                    length_scale_bounds=(1, 50))
    else:
        raise ValueError(f"Unknown kernel: {choice}")
    return k + WK

gp = GaussianProcessRegressor(
    kernel=build_kernel(KERNEL_CHOICE),
    n_restarts_optimizer=N_RESTARTS,
    normalize_y=False,
    random_state=42
)
print(f"Fitting GP with kernel: {KERNEL_CHOICE} ...")
gp.fit(X_train, y_scaled)
print(f"Optimised kernel:\n  {gp.kernel_}")
print(f"\nLog-marginal-likelihood: {gp.log_marginal_likelihood_value_:.3f}")

---
## 7. Forecast

In [ ]:
# Build prediction grid
x_max  = X_train.max()
x_pred = np.linspace(0, x_max + FORECAST_DAYS, 300).reshape(-1, 1)
dates_pred = pd.to_datetime([t0 + pd.Timedelta(days=float(x)) for x in x_pred.ravel()])

# Posterior mean and std
mu_scaled, sigma_scaled = gp.predict(x_pred, return_std=True)
mu    = mu_scaled    * y_std + y_mean
sigma = sigma_scaled * y_std
ci_lo = mu - CI_Z * sigma
ci_hi = mu + CI_Z * sigma

last_observed = series['date'].max()
is_future = dates_pred > last_observed

# Plot
fig, ax = plt.subplots(figsize=(15, 5))
ax.fill_between(dates_pred[~is_future], ci_lo[~is_future], ci_hi[~is_future],
                alpha=0.2, color='royalblue', label=f'{CI_COVERAGE*100:.0f}% CI (fit)')
ax.fill_between(dates_pred[is_future], ci_lo[is_future], ci_hi[is_future],
                alpha=0.25, color='darkorange', label=f'{CI_COVERAGE*100:.0f}% CI (forecast)')
ax.plot(dates_pred[~is_future], mu[~is_future], '-', color='royalblue', lw=2, label='GP mean (fit)')
ax.plot(dates_pred[is_future],  mu[is_future],  '--', color='darkorange', lw=2, label='GP mean (forecast)')
ax.scatter(series['date'], series['y'], color='black', s=30, zorder=5, label=f'Observed ({AGGREGATE})')
ax.axvline(last_observed, color='grey', ls=':', lw=1.5, label='Forecast start')
ax.set_title(f'GP Forecast: Daily {AGGREGATE.upper()} Cycle Time  [{KERNEL_CHOICE} kernel]')
ax.set_ylabel('Cycle Time (s)')
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 8. Anomaly Detection

In [ ]:
# Z-score each observed point against the GP posterior
mu_obs_s, sig_obs_s = gp.predict(X_train, return_std=True)
mu_obs  = mu_obs_s  * y_std + y_mean
sig_obs = sig_obs_s * y_std

series['mu_gp']    = mu_obs
series['sigma_gp'] = sig_obs
series['residual'] = y_train - mu_obs
series['z_score']  = series['residual'] / (sig_obs + 1e-9)
series['anomaly']  = series['z_score'].abs() > ANOMALY_Z_THRESHOLD
series['dow']      = series['date'].dt.day_name()

anomalies = series[series['anomaly']]
print(f"Anomalies (|z| > {ANOMALY_Z_THRESHOLD}): {len(anomalies)}")
if len(anomalies):
    print(anomalies[['date','dow','y','mu_gp','residual','z_score']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 9))

ci_lo_obs = mu_obs - CI_Z * sig_obs
ci_hi_obs = mu_obs + CI_Z * sig_obs

ax = axes[0]
ax.fill_between(series['date'], ci_lo_obs, ci_hi_obs,
                alpha=0.2, color='royalblue', label=f'{CI_COVERAGE*100:.0f}% expected band')
ax.plot(series['date'], mu_obs, '-', color='royalblue', lw=1.5, label='GP mean')
normal = series[~series['anomaly']]
ax.scatter(normal['date'],    normal['y'],    color='steelblue', s=40, zorder=4, label='Normal')
ax.scatter(anomalies['date'], anomalies['y'], color='red', s=80, zorder=5,
           marker='D', label=f'Anomaly (|z|>{ANOMALY_Z_THRESHOLD})')
for _, row in anomalies.iterrows():
    ax.annotate(f"{row['date'].strftime('%b %d')} z={row['z_score']:.1f}",
                xy=(row['date'], row['y']),
                xytext=(5, 8), textcoords='offset points',
                fontsize=8, color='red', fontweight='bold')
ax.set_title(f'Anomaly Detection: GP Expected Band vs Observed  (z-threshold={ANOMALY_Z_THRESHOLD})')
ax.set_ylabel('Cycle Time (s)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

ax2 = axes[1]
colors = ['red' if a else 'steelblue' for a in series['anomaly']]
ax2.bar(series['date'], series['z_score'], color=colors, alpha=0.8, width=0.8)
ax2.axhline( ANOMALY_Z_THRESHOLD, color='red', ls='--', lw=1.5, label=f'+{ANOMALY_Z_THRESHOLD} sigma')
ax2.axhline(-ANOMALY_Z_THRESHOLD, color='red', ls='--', lw=1.5, label=f'-{ANOMALY_Z_THRESHOLD} sigma')
ax2.axhline(0, color='grey', lw=0.8)
ax2.set_title('Standardised Residuals (z-scores)')
ax2.set_ylabel('z-score')
ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 9. Kernel Comparison
Compare all kernels side-by-side. Higher log-ML = better fit.

In [ ]:
kernels_to_compare = ['rbf', 'matern32', 'matern52', 'periodic_rbf', 'rq']
fig, axes = plt.subplots(len(kernels_to_compare), 1,
                          figsize=(14, 4 * len(kernels_to_compare)), sharex=True)

for ax, kname in zip(axes, kernels_to_compare):
    _gp = GaussianProcessRegressor(kernel=build_kernel(kname), n_restarts_optimizer=3, random_state=42)
    _gp.fit(X_train, y_scaled)
    _mu_s, _sig_s = _gp.predict(x_pred, return_std=True)
    _mu  = _mu_s  * y_std + y_mean
    _sig = _sig_s * y_std
    _lml = _gp.log_marginal_likelihood_value_
    ax.fill_between(dates_pred, _mu - CI_Z*_sig, _mu + CI_Z*_sig, alpha=0.25)
    ax.plot(dates_pred, _mu, lw=2)
    ax.scatter(series['date'], series['y'], color='black', s=20, zorder=5)
    ax.axvline(last_observed, color='grey', ls=':', lw=1)
    ax.set_title(f'{kname}  (log-ML = {_lml:.2f})', fontsize=11)
    ax.set_ylabel('Cycle Time (s)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.suptitle('Kernel Comparison', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 10. Custom Date Window
Zoom into any specific period — e.g., a single week, just weekdays, month-end.

In [ ]:
# =====================================================
#   CUSTOM WINDOW — EDIT THESE DATES
# =====================================================
WINDOW_START = '2026-02-05'
WINDOW_END   = '2026-03-09'
# =====================================================

mask_w = (df_raw['date'] >= WINDOW_START) & (df_raw['date'] <= WINDOW_END)
df_w   = df_raw[mask_w].copy()
if OUTLIER_CUTOFF_S:
    df_w = df_w[df_w['cycle_time_s'] <= OUTLIER_CUTOFF_S]

series_w = df_w.groupby('date')['cycle_time_s'].apply(agg_funcs[AGGREGATE]).reset_index()
series_w.columns = ['date', 'y']
series_w = series_w.sort_values('date').reset_index(drop=True)

t0_w = series_w['date'].min()
series_w['x'] = (series_w['date'] - t0_w).dt.days.astype(float)
Xw = series_w['x'].values.reshape(-1, 1)
yw = series_w['y'].values
yw_mean, yw_std = yw.mean(), yw.std()
yw_s = (yw - yw_mean) / yw_std if NORMALISE_Y else yw

gp_w = GaussianProcessRegressor(kernel=build_kernel(KERNEL_CHOICE),
                                  n_restarts_optimizer=N_RESTARTS, random_state=42)
gp_w.fit(Xw, yw_s)

xw_pred = np.linspace(0, Xw.max() + FORECAST_DAYS, 200).reshape(-1, 1)
dw_pred = pd.to_datetime([t0_w + pd.Timedelta(days=float(x)) for x in xw_pred.ravel()])
mu_w_s, sig_w_s = gp_w.predict(xw_pred, return_std=True)
mu_w  = mu_w_s  * (yw_std if NORMALISE_Y else 1) + (yw_mean if NORMALISE_Y else 0)
sig_w = sig_w_s * (yw_std if NORMALISE_Y else 1)

mu_w_obs_s, sig_w_obs_s = gp_w.predict(Xw, return_std=True)
mu_w_obs  = mu_w_obs_s  * (yw_std if NORMALISE_Y else 1) + (yw_mean if NORMALISE_Y else 0)
sig_w_obs = sig_w_obs_s * (yw_std if NORMALISE_Y else 1)
zw = (yw - mu_w_obs) / (sig_w_obs + 1e-9)
anom_w = np.abs(zw) > ANOMALY_Z_THRESHOLD

last_w = series_w['date'].max()
is_fut_w = dw_pred > last_w

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(dw_pred[~is_fut_w], mu_w[~is_fut_w]-CI_Z*sig_w[~is_fut_w],
                mu_w[~is_fut_w]+CI_Z*sig_w[~is_fut_w], alpha=0.2, color='royalblue')
ax.fill_between(dw_pred[is_fut_w], mu_w[is_fut_w]-CI_Z*sig_w[is_fut_w],
                mu_w[is_fut_w]+CI_Z*sig_w[is_fut_w], alpha=0.25, color='darkorange')
ax.plot(dw_pred[~is_fut_w], mu_w[~is_fut_w], '-', color='royalblue', lw=2, label='GP fit')
ax.plot(dw_pred[is_fut_w],  mu_w[is_fut_w],  '--', color='darkorange', lw=2, label='Forecast')
ax.scatter(series_w['date'][~anom_w], yw[~anom_w], color='steelblue', s=50, zorder=4)
ax.scatter(series_w['date'][anom_w],  yw[anom_w],  color='red', marker='D', s=80, zorder=5,
           label=f'Anomaly |z|>{ANOMALY_Z_THRESHOLD}')
ax.axvline(last_w, color='grey', ls=':', lw=1.5)
ax.set_title(f'Custom Window: {WINDOW_START} to {WINDOW_END}  [{KERNEL_CHOICE}, {AGGREGATE}]')
ax.set_ylabel('Cycle Time (s)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
if anom_w.any():
    ax.legend()
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()
print(f"Window: {len(series_w)} points, {anom_w.sum()} anomalies")

---
## 11. Tuning Guide

| What you want | What to change |
|---|---|
| Zoom into a date range | DATE_START / DATE_END in Section 4 |
| Different statistic | AGGREGATE -> 'mean', 'median', 'p95', 'p99' |
| Smoother / rougher fit | KERNEL_CHOICE -> 'rbf' (smooth) or 'matern32' (rough) |
| Capture weekly seasonality | KERNEL_CHOICE = 'periodic_rbf' |
| More / fewer anomaly flags | ANOMALY_Z_THRESHOLD up=fewer, down=more |
| Wider / narrower CI band | CI_COVERAGE |
| Extend forecast | FORECAST_DAYS |
| Better fit (slower) | N_RESTARTS up |
| Drop extreme outliers | OUTLIER_CUTOFF_S |
| Focus on a specific window | WINDOW_START / WINDOW_END in Section 10 |

---

### What the EDA shows in this dataset

- **133k records, 35 days** (Feb 2 - Mar 9 2026). One Sunday missing.
- **Bulk of cycle times: 15-16s** (very tight normal operating range), fat tail to 544s.
- **p99 is consistently ~29s** across almost all days — a stable slow-path for ~1% of jobs.
- **Saturdays** have elevated mean and p95 — likely different shift or staffing.
- **Feb 23 (Mon) and Mar 4 (Wed)** show elevated means — possible production incidents.
- **Sundays** have much lower volume (100-1900 records vs 3000-5000 on weekdays).

### Suggested next steps
- Try `AGGREGATE = 'p95'` to monitor tail latency instead of average
- Try `KERNEL_CHOICE = 'periodic_rbf'` to explicitly model the weekly pattern
- Filter to weekdays only (exclude weekend noise) and compare to the full fit
- Subset to individual machine IDs to detect per-machine degradation